In [ ]:
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from model.devanagari_model import DevanagariCNN

TEST_DIR = Path("data/DevanagariHandwrittenCharacterDataset/Test")
WEIGHTS_PATH = Path("hindi_cnn_weights_pytorch.pt")
ONNX_PATH = Path("hindi_cnn.onnx")

NUM_IMAGES = 100
IMG_SIZE = (32, 32)

In [2]:
# ---------------------------------------------------------------------
# Load PyTorch model
# ---------------------------------------------------------------------

model = DevanagariCNN()

state_dict = torch.load(
    WEIGHTS_PATH,
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(state_dict)
model.eval()

# ---------------------------------------------------------------------
# Load ONNX model
# ---------------------------------------------------------------------

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

input_name = session.get_inputs()[0].name

In [3]:
# ---------------------------------------------------------------------
# Load test dataset
# ---------------------------------------------------------------------

transform = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
    ]
)

dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
)

In [4]:
# ---------------------------------------------------------------------
# Compare predictions
# ---------------------------------------------------------------------

matches = 0
total = 0

for image, _ in loader:

    # PyTorch prediction
    with torch.no_grad():
        pytorch_logits = model(image)
        pytorch_pred = pytorch_logits.argmax(dim=1).item()

    # ONNX prediction
    onnx_logits = session.run(
        None,
        {input_name: image.numpy()},
    )[0]

    onnx_pred = int(np.argmax(onnx_logits, axis=1)[0])

    if pytorch_pred == onnx_pred:
        matches += 1

    total += 1

    if total >= NUM_IMAGES:
        break

agreement = matches / total

print(f"Compared {total} images")
print(f"Matching predictions: {matches}")
print(f"Agreement: {agreement:.2%}")

if agreement >= 0.99:
    print("ONNX export validated.")
else:
    print("Agreement below 99%.")

Compared 100 images
Matching predictions: 100
Agreement: 100.00%
ONNX export validated.
